<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/02_Intermediate/L17_Fine_tuning_Techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L17: Fine-tuning Techniques - Training Loops and Optimization

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Intermediate  
**Lesson:** 17 of 30

---

## Learning Objectives

By the end of this lesson, you will:
- Set up full fine-tuning with HuggingFace Trainer and custom loops
- Use appropriate loss functions for classification and generation
- Apply learning rate schedules and gradient clipping
- Monitor training metrics and prevent overfitting
- Use gradient checkpointing and mixed precision for efficiency
- Tune key hyperparameters (LR, batch size, epochs)

---

## Concept: Fine-tuning in Practice

**Full fine-tuning** updates all (or most) parameters of a pre-trained model on your task. Key choices:
- **Loss**: Cross-entropy for classification; language modeling loss for generation.
- **Optimizer**: AdamW with weight decay is standard.
- **Learning rate**: Small (e.g. 1e-5 to 5e-5) to avoid forgetting.
- **Schedules**: Linear warmup + decay often helps.
- **Regularization**: Weight decay, dropout, early stopping.
- **Efficiency**: Gradient checkpointing (trade compute for memory), mixed precision (FP16/BF16).

---

In [ ]:
# Step 1: Install and Import

!pip install transformers torch accelerate datasets -q

import torch
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, get_linear_schedule_with_warmup
)
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Part 1: Data and Model Setup

We use a small classification dataset and DistilBERT. The same patterns apply to other encoders and to generation models.

---

In [ ]:
# Step 2: Load Data and Tokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

train_ds = load_dataset("imdb", split="train", trust_remote_code=True).select(range(800))
eval_ds = load_dataset("imdb", split="test", trust_remote_code=True).select(range(200))

train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
eval_ds = eval_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
train_ds.set_format("torch")
eval_ds.set_format("torch")

print(f"Train: {len(train_ds)}, Eval: {len(eval_ds)}")

## Part 2: Fine-tuning with Trainer (Recommended)

The **Trainer** handles the training loop, loss, optimizer, LR schedule, logging, and evaluation. You configure everything via `TrainingArguments`.

---

In [ ]:
# Step 3: TrainingArguments - Loss, Optimizer, Schedule

training_args = TrainingArguments(
    output_dir="./out_finetune_l17",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    logging_steps=25,
    report_to="none",
)

print("Key settings:")
print(f"  LR: {training_args.learning_rate}, weight_decay: {training_args.weight_decay}")
print(f"  warmup_ratio: {training_args.warmup_ratio}, max_grad_norm: {training_args.max_grad_norm}")
print(f"  fp16: {training_args.fp16}")

In [ ]:
# Step 4: Train with Trainer

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
)

trainer.train()
metrics = trainer.evaluate()
print("Eval metrics:", metrics)

## Part 3: Custom Training Loop (Loss and Optimizer)

A **manual loop** gives full control over loss, optimizer, and logging. Here we use cross-entropy (built into the model's forward) and AdamW with a linear warmup schedule.

---

In [ ]:
# Step 5: Custom Loop - Loss, Optimizer, Schedule

from torch.optim import AdamW

model_custom = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_custom = model_custom.to(device)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
num_training_steps = len(train_loader) * 2
optimizer = AdamW(model_custom.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * num_training_steps), num_training_steps=num_training_steps)

model_custom.train()
for epoch in range(2):
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model_custom(**batch)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_custom.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        if (step + 1) % 50 == 0:
            print(f"Epoch {epoch+1} step {step+1} loss: {loss.item():.4f}")
    print(f"Epoch {epoch+1} avg loss: {total_loss/len(train_loader):.4f}")

print("\nCustom loop done. Loss = model output (cross-entropy for classification).")

## Part 4: Gradient Checkpointing (Save Memory)

**Gradient checkpointing** recomputes activations in the backward pass instead of storing them. Use it when you hit OOM; training becomes slower but uses less memory.

---

In [ ]:
# Step 6: Enable Gradient Checkpointing

model_ckpt = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
if hasattr(model_ckpt, "gradient_checkpointing_enable"):
    model_ckpt.gradient_checkpointing_enable()
    print("Gradient checkpointing enabled (saves memory, slightly slower).")
else:
    print("This model does not support gradient checkpointing.")

# Trainer also supports it via args:
# TrainingArguments(..., gradient_checkpointing=True)

## Part 5: Overfitting and Early Stopping

**Overfitting** appears when train loss keeps decreasing while eval loss increases. Mitigations: more data, dropout, weight decay, fewer epochs, **early stopping** (stop when eval metric stops improving).

---

In [ ]:
# Step 7: Early Stopping with Trainer (via callbacks)

from transformers import EarlyStoppingCallback

args_early = TrainingArguments(
    output_dir="./out_early_stop_l17",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

model_es = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
trainer_es = Trainer(
    model=model_es,
    args=args_early,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer_es.train()
print("Training stopped early if eval loss did not improve for 2 epochs.")

## Part 6: Hyperparameter Summary

| Parameter       | Typical range / note |
|----------------|----------------------|
| Learning rate  | 1e-5 to 5e-5 for BERT-style |
| Batch size     | 8–32 (larger with gradient accumulation) |
| Epochs         | 2–5; use early stopping |
| Warmup ratio   | 0.05–0.1 |
| Weight decay   | 0.01 |
| Max grad norm  | 1.0 |
| FP16/BF16      | Use when GPU supports it |

---

## Exercises

### Exercise 1: Learning Rate Sweep
Train with LR = 1e-5, 2e-5, 5e-5 (same epochs and data). Log train and eval loss each epoch. Which LR gives the best final eval loss?

### Exercise 2: Batch Size and Gradient Accumulation
Simulate a large effective batch size: set per_device_train_batch_size=4 and gradient_accumulation_steps=4 (effective batch = 16). Compare training time and final metric with batch_size=16 and no accumulation.

### Exercise 3: Warmup Ablation
Train with warmup_ratio=0 and warmup_ratio=0.15. Compare stability (first-epoch loss) and final accuracy.

### Exercise 4: Generation Task
Fine-tune a small causal LM (e.g. GPT-2) on a few hundred lines of domain text using the Trainer with a language-modeling task. Monitor perplexity.

### Exercise 5: Logging and Plotting
Use TensorBoard or a simple list to record loss per step. Plot train vs eval loss over time and identify overfitting (eval loss going up while train loss goes down).

---

## Key Takeaways

1. **Trainer** handles loss, AdamW, LR schedule, clipping, and evaluation; use it unless you need a custom loop.
2. **Loss**: classification uses cross-entropy (from model output); generation uses LM loss.
3. **Small LR + warmup + weight decay** are standard for fine-tuning.
4. **Gradient checkpointing** reduces memory; **FP16/BF16** speeds up and saves memory.
5. **Early stopping** and monitoring eval metrics help prevent overfitting.

---

## Additional Resources

- [HuggingFace Training](https://huggingface.co/docs/transformers/training), [TrainingArguments](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments)
- [AdamW](https://arxiv.org/abs/1711.05101), [Warmup](https://arxiv.org/abs/1706.02677)

---

## Next Lesson

**L18: LoRA Adapters** – Low-rank adaptation, parameter-efficient fine-tuning with the PEFT library, and comparing LoRA to full fine-tuning.

---